# Leaves State Classifier (SIT only): Fresh vs Mature vs Old

Goal: label each tracked SIT tree crown at each orthomosaic date as one of:
- **LeavesFresh**: light green (new flush)
- **LeavesMature**: full green canopy
- **LeavesOld**: yellowing / senescent / shedding / sparse canopy

> This notebook reuses the **SIT tracking outputs** (consensus crown crops + manifests) produced by the crown tracking pipeline (`crown_tracking_31Mar26.ipynb`).

> We focus on classifier quality + visual audits, so you can iterate on thresholds/weights quickly.

In [1]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Image IO
import imageio.v2 as imageio

# Optional (better texture metrics)
SKIMAGE_AVAILABLE = True
try:
    from skimage.feature import graycomatrix, graycoprops
except Exception:
    SKIMAGE_AVAILABLE = False

# Optional (used only for a few ops; notebook still works without)
CV2_AVAILABLE = True
try:
    import cv2
except Exception:
    CV2_AVAILABLE = False

print('skimage texture available:', SKIMAGE_AVAILABLE)
print('cv2 available:', CV2_AVAILABLE)

# ── Paths / Dataset ───────────────────────────────────────────────────────────
ROOT = Path.cwd().resolve().parents[1]  # repo root when running from src/notebooks
DATASET = 'SIT'
RUN_TAG = 'sit'

SIT_RUN_DIR = ROOT / 'output' / 'sit_tracking_rerun_31Mar26'
CROPS_DIR = SIT_RUN_DIR / 'consensus_crown_crops_all_oms'
MANIFEST_CSV = CROPS_DIR / 'consensus_crops_manifest.csv'
PHENO_DIR = SIT_RUN_DIR / 'phenology'

# Canonical inputs we will consume
FEATURES_CSV_FALLBACK = PHENO_DIR / 'phenology_features_raw_tree_only.csv'  # may exist already
NON_TREE_DROPS_CSV = PHENO_DIR / 'non_tree_dropped_crowns.csv'  # if present

assert SIT_RUN_DIR.exists(), f'Missing SIT run dir: {SIT_RUN_DIR}'
assert MANIFEST_CSV.exists(), f'Missing crops manifest: {MANIFEST_CSV}'

print('ROOT:', ROOT)
print('SIT_RUN_DIR:', SIT_RUN_DIR)
print('CROPS_DIR:', CROPS_DIR)
print('PHENO_DIR:', PHENO_DIR)
print('MANIFEST_CSV:', MANIFEST_CSV)

skimage texture available: True
cv2 available: True
ROOT: /Users/hbot07/VS Code/Drone-Phenology-Monitoring
SIT_RUN_DIR: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_31Mar26
CROPS_DIR: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_31Mar26/consensus_crown_crops_all_oms
PHENO_DIR: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_31Mar26/phenology
MANIFEST_CSV: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sit_tracking_rerun_31Mar26/consensus_crown_crops_all_oms/consensus_crops_manifest.csv


## 1) Load SIT crop manifest
The manifest maps each consensus crown (`chain_id`) to its per-OM crop PNGs.

In [2]:
manifest = pd.read_csv(MANIFEST_CSV)
manifest['chain_id'] = manifest['consensus_index'].astype(int) + 1  # convention: chain_id is 1-based
manifest['om_id'] = manifest['om_id'].astype(int)
manifest['crop_path'] = manifest['crop_path'].astype(str)

# Keep only successful crops
manifest_ok = manifest[manifest['status'] == 'ok'].copy()
om_ids = sorted(manifest_ok['om_id'].unique().tolist())
n_oms = len(om_ids)

print('Crops rows (ok):', len(manifest_ok))
print('Trees (unique chain_id):', manifest_ok['chain_id'].nunique())
print('OM ids:', om_ids)

# Optional: filter out non-trees if drop-list exists
non_tree_ids = set()
if NON_TREE_DROPS_CSV.exists():
    drops = pd.read_csv(NON_TREE_DROPS_CSV)
    if 'chain_id' in drops.columns:
        non_tree_ids = set(pd.to_numeric(drops['chain_id'], errors='coerce').dropna().astype(int).tolist())
        print('Non-tree drop-list loaded:', len(non_tree_ids))
    else:
        print('Non-tree CSV present but missing chain_id column:', NON_TREE_DROPS_CSV)
else:
    print('Non-tree drop-list not found (ok):', NON_TREE_DROPS_CSV)

manifest_tree = manifest_ok[~manifest_ok['chain_id'].isin(non_tree_ids)].copy()
print('After non-tree filtering:')
print('  rows:', len(manifest_tree))
print('  trees:', manifest_tree['chain_id'].nunique())
print('  dropped trees:', len(non_tree_ids))

Crops rows (ok): 3102
Trees (unique chain_id): 222
OM ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Non-tree drop-list loaded: 5
After non-tree filtering:
  rows: 3032
  trees: 217
  dropped trees: 5


## 2) Feature extraction from crop PNGs
We recompute rich phenology/texture/color signals from the crop images so the leaf-state classifier can use: GCC/RCC/ExG, green/yellow fractions, brightness/saturation, entropy, Laplacian sharpness, and (optionally) GLCM texture.

In [ ]:
def _safe_div(num: np.ndarray, den: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    return num / (den + eps)


def _shannon_entropy_01(values_01: np.ndarray, bins: int = 64) -> float:
    """Probability-based Shannon entropy for values expected in [0,1]."""
    if values_01.size == 0:
        return float('nan')
    hist, _ = np.histogram(values_01, bins=bins, range=(0, 1), density=False)
    total = hist.sum()
    if total <= 0:
        return float('nan')
    p = hist.astype(np.float64) / total
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


def _rgb_to_hsv01(rgb01: np.ndarray) -> np.ndarray:
    """rgb01: float32 in [0,1], returns hsv in [0,1]."""
    # vectorized conversion (avoid matplotlib dependency)
    r = rgb01[..., 0]
    g = rgb01[..., 1]
    b = rgb01[..., 2]
    mx = np.maximum(np.maximum(r, g), b)
    mn = np.minimum(np.minimum(r, g), b)
    diff = mx - mn
    h = np.zeros_like(mx)
    s = np.zeros_like(mx)
    v = mx
    # saturation
    s[mx > 0] = diff[mx > 0] / (mx[mx > 0] + 1e-12)
    # hue
    mask = diff > 1e-12
    # where max is r
    m = mask & (mx == r)
    h[m] = ((g[m] - b[m]) / diff[m]) % 6
    # where max is g
    m = mask & (mx == g)
    h[m] = ((b[m] - r[m]) / diff[m]) + 2
    # where max is b
    m = mask & (mx == b)
    h[m] = ((r[m] - g[m]) / diff[m]) + 4
    h = (h / 6.0) % 1.0
    hsv = np.stack([h, s, v], axis=-1)
    return hsv


def compute_features_from_rgb_u8(patch_u8: np.ndarray) -> Dict[str, float]:
    """Compute rich signals from a uint8 RGB(A) patch with nodata=0 background."""
    if patch_u8 is None or patch_u8.size == 0:
        return {}
    if patch_u8.ndim != 3 or patch_u8.shape[2] < 3:
        return {}
    rgb_u8 = patch_u8[..., :3]
    rgb = rgb_u8.astype(np.float32) / 255.0
    finite_mask = np.isfinite(rgb).all(axis=2)
    nonzero_mask = (rgb_u8.sum(axis=2) > 0)
    valid = finite_mask & nonzero_mask
    total_px = int(rgb.shape[0] * rgb.shape[1])
    valid_px = int(valid.sum())
    if total_px == 0:
        return {}
    if valid_px < 30:
        return {'valid_pixel_fraction': valid_px / max(total_px, 1)}
    r = rgb[..., 0]
    g = rgb[..., 1]
    b = rgb[..., 2]
    denom = (r + g + b)
    gcc = _safe_div(g, denom)
    rcc = _safe_div(r, denom)
    exg = (2.0 * g - r - b)

    hsv = _rgb_to_hsv01(rgb)
    h = hsv[..., 0]
    s = hsv[..., 1]
    v = hsv[..., 2]

    # Masks (tuneable; chosen to separate green vs yellow reasonably well)
    dark = (v < 0.12)
    green = valid & (h >= 0.18) & (h <= 0.48) & (s >= 0.15) & (v >= 0.12)
    yellow = valid & (h >= 0.08) & (h <= 0.18) & (s >= 0.18) & (v >= 0.18)
    low_sat = valid & (s < 0.12) & (v >= 0.15)

    gray = (0.299 * r + 0.587 * g + 0.114 * b)
    if CV2_AVAILABLE:
        gray_smooth = cv2.GaussianBlur(gray, (5, 5), 0)
        gray_u8 = np.clip(gray * 255, 0, 255).astype(np.uint8)
        lap_var = float(cv2.Laplacian(gray_u8, cv2.CV_64F).var())
    else:
        gray_smooth = gray
        lap_var = float('nan')

    vals_gray = gray[valid]
    vals_gray_s = gray_smooth[valid]
    vals_r = r[valid]
    vals_g = g[valid]
    vals_b = b[valid]
    vals_gcc = gcc[valid]
    vals_rcc = rcc[valid]
    vals_exg = exg[valid]
    vals_s = s[valid]
    vals_v = v[valid]

    feats: Dict[str, float] = {
        'valid_pixel_fraction': float(valid_px / total_px),
        'shadow_fraction': float((dark & valid).sum() / max(valid_px, 1)),
        'veg_fraction_hsv': float(green.sum() / max(valid_px, 1)),
        'yellow_fraction_hsv': float(yellow.sum() / max(valid_px, 1)),
        'low_saturation_fraction': float(low_sat.sum() / max(valid_px, 1)),
        'gcc_mean': float(np.mean(vals_gcc)),
        'rcc_mean': float(np.mean(vals_rcc)),
        'exg_mean': float(np.mean(vals_exg)),
        'r_mean': float(np.mean(vals_r)),
        'g_mean': float(np.mean(vals_g)),
        'b_mean': float(np.mean(vals_b)),
        'r_std': float(np.std(vals_r)),
        'g_std': float(np.std(vals_g)),
        'b_std': float(np.std(vals_b)),
        'gray_std': float(np.std(vals_gray)),
        'gray_std_smooth': float(np.std(vals_gray_s)),
        'gray_entropy': float(_shannon_entropy_01(vals_gray)),
        'hsv_s_mean': float(np.mean(vals_s)),
        'hsv_v_mean': float(np.mean(vals_v)),
        'laplacian_var': float(lap_var) if np.isfinite(lap_var) else float('nan'),
    }

    # Optional: GLCM texture on quantized gray
    if SKIMAGE_AVAILABLE:
        try:
            quant = np.clip((gray * 31).astype(np.uint8), 0, 31)
            glcm = graycomatrix(
                quant,
                distances=[1],
                angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                levels=32,
                symmetric=True,
                normed=True,
            )
            feats['glcm_contrast'] = float(graycoprops(glcm, 'contrast').mean())
            feats['glcm_homogeneity'] = float(graycoprops(glcm, 'homogeneity').mean())
            feats['glcm_energy'] = float(graycoprops(glcm, 'energy').mean())
        except Exception:
            feats['glcm_contrast'] = float('nan')
            feats['glcm_homogeneity'] = float('nan')
            feats['glcm_energy'] = float('nan')
    else:
        feats['glcm_contrast'] = float('nan')
        feats['glcm_homogeneity'] = float('nan')
        feats['glcm_energy'] = float('nan')

    # QC flag (same spirit as your leafshed notebook)
    feats['is_bad_observation'] = bool(
        (feats['valid_pixel_fraction'] < 0.60)
        or (feats['shadow_fraction'] > 0.60)
        or (np.isfinite(feats.get('laplacian_var', np.nan)) and feats['laplacian_var'] < 25)
    )
    return feats


def read_rgb_u8(path: str) -> Optional[np.ndarray]:
    try:
        arr = imageio.imread(path)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.ndim == 3 and arr.shape[2] >= 3:
            return arr[..., :3].astype(np.uint8)
        return None
    except Exception:
        return None


print('✓ Feature extraction helpers ready')

In [ ]:
# Compute features for all (tree, OM) crops
# - Set MAX_TREES to a small number first for quick iteration
MAX_TREES: Optional[int] = 40   # None = all
RECOMPUTE_FEATURES = True
FEATURES_CACHE_CSV = PHENO_DIR / 'leaves_state_features_rich.csv'

if (not RECOMPUTE_FEATURES) and FEATURES_CACHE_CSV.exists():
    feats_df = pd.read_csv(FEATURES_CACHE_CSV)
    feats_df['chain_id'] = feats_df['chain_id'].astype(int)
    feats_df['om_id'] = feats_df['om_id'].astype(int)
    print('Loaded cached rich features:', FEATURES_CACHE_CSV)
else:
    tree_ids = sorted(manifest_tree['chain_id'].unique().tolist())
    if MAX_TREES is not None:
        tree_ids = tree_ids[: int(MAX_TREES)]
    sub = manifest_tree[manifest_tree['chain_id'].isin(tree_ids)].copy()
    sub = sub.sort_values(['chain_id', 'om_id']).reset_index(drop=True)

    rows = []
    for i, r in enumerate(sub.itertuples(index=False), start=1):
        rgb_u8 = read_rgb_u8(r.crop_path)
        feats = compute_features_from_rgb_u8(rgb_u8)
        row = {
            'chain_id': int(r.chain_id),
            'om_id': int(r.om_id),
            'date_label': str(r.stem),
            'patch_exists': True,
            'patch_h': int(r.height_px) if hasattr(r, 'height_px') else (int(rgb_u8.shape[0]) if rgb_u8 is not None else np.nan),
            'patch_w': int(r.width_px) if hasattr(r, 'width_px') else (int(rgb_u8.shape[1]) if rgb_u8 is not None else np.nan),
        }
        row.update(feats)
        rows.append(row)
        if i % 250 == 0:
            print(f'  processed {i}/{len(sub)} crops...')

    feats_df = pd.DataFrame(rows).sort_values(['chain_id', 'om_id']).reset_index(drop=True)
    FEATURES_CACHE_CSV.parent.mkdir(parents=True, exist_ok=True)
    feats_df.to_csv(FEATURES_CACHE_CSV, index=False)
    print('Saved rich features cache:', FEATURES_CACHE_CSV)

print('Rich feature rows:', len(feats_df))
print('Trees:', feats_df['chain_id'].nunique())
print('OMs:', sorted(feats_df['om_id'].unique().tolist()))
display(feats_df.head())

## 3) Normalization (robust by date) + derived indices
We compute robust per-date z-scores (median/MAD) so the classifier is less sensitive to global illumination changes between flights.

In [ ]:
norm_df = feats_df.copy()
norm_df['is_bad_observation'] = norm_df.get('is_bad_observation', False).fillna(True).astype(bool)

# Derived indices helpful for leaf states
norm_df['yellow_minus_green'] = (norm_df.get('yellow_fraction_hsv', np.nan) - norm_df.get('veg_fraction_hsv', np.nan)).astype(float)
norm_df['green_minus_red'] = (norm_df.get('gcc_mean', np.nan) - norm_df.get('rcc_mean', np.nan)).astype(float)
norm_df['red_minus_green'] = -norm_df['green_minus_red']

# Robust z-score per OM (median/MAD)
def robust_z_by_group(df: pd.DataFrame, col: str, group_col: str = 'om_id') -> pd.Series:
    x = pd.to_numeric(df[col], errors='coerce')
    med = x.groupby(df[group_col]).transform('median')
    mad = x.groupby(df[group_col]).transform(lambda s: np.median(np.abs(s - np.median(s))))
    scale = 1.4826 * mad.replace(0, np.nan)
    return (x - med) / (scale + 1e-8)

FEATURE_COLS = [
    'gcc_mean','rcc_mean','exg_mean','veg_fraction_hsv','yellow_fraction_hsv',
    'hsv_s_mean','hsv_v_mean',
    'gray_std_smooth','gray_entropy','laplacian_var',
    'glcm_contrast','glcm_homogeneity','glcm_energy',
    'yellow_minus_green','green_minus_red','red_minus_green',
    'shadow_fraction','valid_pixel_fraction','low_saturation_fraction',
 ]
FEATURE_COLS = [c for c in FEATURE_COLS if c in norm_df.columns]
print('Feature columns used:', FEATURE_COLS)

for c in FEATURE_COLS:
    norm_df[f'{c}_rz_date'] = robust_z_by_group(norm_df, c, group_col='om_id')
    norm_df[f'{c}_pct_date'] = pd.to_numeric(norm_df[c], errors='coerce').groupby(norm_df['om_id']).rank(pct=True)

# Keep a clean subset for some plots (classifier still handles bad obs)
clean_df = norm_df[~norm_df['is_bad_observation']].copy()
print('Clean obs:', len(clean_df), '/', len(norm_df), f'({len(clean_df)/max(len(norm_df),1):.1%})')

display(norm_df[['om_id','is_bad_observation']].value_counts().rename('n').to_frame().reset_index().sort_values(['om_id','is_bad_observation']))

## 4) LeavesFresh / LeavesMature / LeavesOld classifier (rule-based + smoothing)
We start with a strong heuristic model that uses **multiple signals** (green/yellow fractions, GCC/RCC/ExG, texture/entropy, saturation/brightness).
Then we apply a simple dynamic-programming smoother per tree to enforce plausible seasonal transitions.

In [ ]:
LEAF_STATES = ['LeavesFresh', 'LeavesMature', 'LeavesOld']

@dataclass(frozen=True)
class LeafStateWeights:
    # Core normalized (rz_date) weights
    w_green: float = 1.10
    w_yellow: float = 1.25
    w_gcc: float = 0.90
    w_rcc: float = 0.75
    w_exg: float = 0.70
    w_entropy: float = 0.55
    w_gray_std: float = 0.35
    w_sat: float = 0.55
    w_val: float = 0.55
    w_shadow: float = 0.25
    # Time prior strength (small, to avoid forcing wrong labels)
    w_time_prior: float = 0.25
    # Bad observation penalty
    bad_obs_penalty: float = 1.10


WEIGHTS = LeafStateWeights()
print('✓ LeafStateWeights ready')


def _get(df: pd.DataFrame, col: str) -> np.ndarray:
    if col not in df.columns:
        return np.zeros(len(df), dtype=float)
    return pd.to_numeric(df[col], errors='coerce').fillna(0.0).to_numpy(dtype=float)


def score_leaf_states(df: pd.DataFrame, weights: LeafStateWeights) -> pd.DataFrame:
    """Return df with raw scores + softmax probabilities for each state."""
    out = df.copy()
    t = out['om_id'].astype(float).to_numpy()
    om_min = float(np.nanmin(t)) if np.isfinite(t).any() else 1.0
    om_max = float(np.nanmax(t)) if np.isfinite(t).any() else om_min + 1.0
    t01 = (t - om_min) / max(om_max - om_min, 1.0)  # 0 early → 1 late
    # Smooth priors: fresh early, old late, mature mid
    prior_fresh = (1.0 - t01)
    prior_old = t01
    prior_mature = 1.0 - np.abs(t01 - 0.5) * 2.0
    prior_mature = np.clip(prior_mature, 0.0, 1.0)

    green = _get(out, 'veg_fraction_hsv_rz_date')
    yellow = _get(out, 'yellow_fraction_hsv_rz_date')
    gcc = _get(out, 'gcc_mean_rz_date')
    rcc = _get(out, 'rcc_mean_rz_date')
    exg = _get(out, 'exg_mean_rz_date')
    ent = _get(out, 'gray_entropy_rz_date')
    gstd = _get(out, 'gray_std_smooth_rz_date')
    sat = _get(out, 'hsv_s_mean_rz_date')
    val = _get(out, 'hsv_v_mean_rz_date')
    shad = _get(out, 'shadow_fraction_rz_date')
    ymg = _get(out, 'yellow_minus_green_rz_date')
    rmg = _get(out, 'red_minus_green_rz_date')

    # LeavesFresh: green + bright, low yellow/red, relatively low texture
    score_fresh = (
        + weights.w_green * green
        + weights.w_gcc * gcc
        + weights.w_exg * exg
        + weights.w_val * val
        - weights.w_sat * sat * 0.45
        - weights.w_yellow * yellow
        - 0.50 * weights.w_rcc * rcc
        - 0.45 * weights.w_entropy * ent
        - 0.35 * weights.w_gray_std * gstd
        - 0.45 * ymg
        - 0.25 * rmg
        - 0.20 * weights.w_shadow * shad
        + weights.w_time_prior * prior_fresh
    )

    # LeavesMature: very green, higher saturation, stable canopy (low yellow), moderate brightness
    score_mature = (
        + 1.15 * weights.w_green * green
        + 0.85 * weights.w_gcc * gcc
        - 0.85 * weights.w_yellow * yellow
        - 0.35 * ymg
        + 0.55 * weights.w_sat * sat
        + 0.20 * weights.w_val * val
        - 0.15 * weights.w_entropy * ent
        - 0.10 * weights.w_gray_std * gstd
        - 0.20 * weights.w_shadow * shad
        + weights.w_time_prior * prior_mature
    )

    # LeavesOld: yellow/red shift, reduced green, increased texture/entropy (branches visible), often more shadow/no-data
    score_old = (
        + weights.w_yellow * yellow
        + 0.85 * weights.w_rcc * rcc
        + 0.55 * weights.w_entropy * ent
        + 0.45 * weights.w_gray_std * gstd
        + 0.55 * ymg
        + 0.35 * rmg
        - 1.00 * weights.w_green * green
        - 0.65 * weights.w_gcc * gcc
        - 0.25 * weights.w_exg * exg
        - 0.10 * weights.w_sat * sat
        + 0.15 * weights.w_shadow * shad
        + weights.w_time_prior * prior_old
    )

    bad = out['is_bad_observation'].astype(bool).to_numpy() if 'is_bad_observation' in out.columns else np.zeros(len(out), dtype=bool)
    penalty = weights.bad_obs_penalty * bad.astype(float)
    score_fresh = score_fresh - penalty
    score_mature = score_mature - penalty
    score_old = score_old - penalty

    out['score_fresh'] = score_fresh
    out['score_mature'] = score_mature
    out['score_old'] = score_old

    S = np.vstack([score_fresh, score_mature, score_old]).T
    # softmax (stable)
    S_max = np.max(S, axis=1, keepdims=True)
    P = np.exp(S - S_max)
    P = P / (np.sum(P, axis=1, keepdims=True) + 1e-12)
    out['p_fresh'] = P[:, 0]
    out['p_mature'] = P[:, 1]
    out['p_old'] = P[:, 2]

    pred_idx = np.argmax(S, axis=1)
    out['leaf_state_raw'] = [LEAF_STATES[i] for i in pred_idx]
    out['confidence_raw'] = np.take_along_axis(P, pred_idx[:, None], axis=1).ravel()
    return out


def smooth_leaf_states_per_tree(df_scored: pd.DataFrame, trans_penalty: float = 0.55) -> pd.DataFrame:
    """
    Viterbi-like smoothing over OM order using log-probabilities + transition penalties.

    Allowed seasonal intuition:
    Fresh → Mature → Old (forward), with self-transitions allowed.
    Backward jumps are penalized harder.
    """
    out = df_scored.copy()
    # transition costs (subtract from log prob): smaller = easier
    # rows: from, cols: to
    base = trans_penalty
    Tcost = np.array([
        [0.00, base * 0.8, base * 2.0],   # Fresh -> (Fresh, Mature, Old)
        [base * 1.2, 0.00, base * 0.8],   # Mature -> (Fresh, Mature, Old)
        [base * 2.2, base * 1.2, 0.00],   # Old -> (Fresh, Mature, Old)
    ], dtype=float)

    smoothed = []
    for chain_id, g in out.groupby('chain_id', sort=True):
        g = g.sort_values('om_id').reset_index(drop=True)
        # avoid log(0)
        P = g[['p_fresh', 'p_mature', 'p_old']].to_numpy(dtype=float)
        P = np.clip(P, 1e-6, 1.0)
        logp = np.log(P)

        n = len(g)
        dp = np.full((n, 3), -np.inf)
        back = np.full((n, 3), -1, dtype=int)
        dp[0, :] = logp[0, :]

        for i in range(1, n):
            for s in range(3):
                best_val = -np.inf
                best_prev = 0
                for sp in range(3):
                    val = dp[i - 1, sp] + logp[i, s] - Tcost[sp, s]
                    if val > best_val:
                        best_val = val
                        best_prev = sp
                dp[i, s] = best_val
                back[i, s] = best_prev

        last = int(np.argmax(dp[n - 1, :]))
        path = [last]
        for i in range(n - 1, 0, -1):
            last = int(back[i, last])
            path.append(last)
        path = list(reversed(path))

        g['leaf_state_smooth'] = [LEAF_STATES[i] for i in path]
        g['confidence_smooth'] = [float(P[i, path[i]]) for i in range(n)]
        smoothed.append(g)

    return pd.concat(smoothed, ignore_index=True)


scored = score_leaf_states(norm_df, WEIGHTS)
pred_df = smooth_leaf_states_per_tree(scored, trans_penalty=0.55)

print('Pred rows:', len(pred_df))
display(pred_df[['chain_id','om_id','leaf_state_raw','confidence_raw','leaf_state_smooth','confidence_smooth']].head(20))

In [ ]:
# Save predictions (so downstream notebooks / manual edits can consume them)
OUT_PRED_CSV = PHENO_DIR / 'leaves_state_predictions_sit.csv'
pred_df.to_csv(OUT_PRED_CSV, index=False)
print('Wrote:', OUT_PRED_CSV)

# Quick counts by OM
counts = (
    pred_df.groupby(['om_id','leaf_state_smooth'])['chain_id']
    .nunique()
    .rename('n_trees')
    .reset_index()
    .sort_values(['om_id','leaf_state_smooth'])
)
display(counts)

## 5) Visual audits (so we can tune the classifier)
These plots are designed for fast iteration: if you tell me which mistakes you see, we’ll adjust weights/thresholds.

In [ ]:
STATE_COLORS = {
    'LeavesFresh': '#7bd389',   # light green
    'LeavesMature': '#1f7a3a',  # deep green
    'LeavesOld': '#e3b448',     # yellow-ish
}

def _scatter_by_state(df: pd.DataFrame, x: str, y: str, title: str, use_smooth: bool = True, om_id: Optional[int] = None):
    col_state = 'leaf_state_smooth' if use_smooth else 'leaf_state_raw'
    sub = df.copy()
    if om_id is not None:
        sub = sub[sub['om_id'] == int(om_id)]
    sub = sub[np.isfinite(pd.to_numeric(sub[x], errors='coerce')) & np.isfinite(pd.to_numeric(sub[y], errors='coerce'))]
    if sub.empty:
        print('No data for scatter:', x, y)
        return
    fig, ax = plt.subplots(figsize=(7, 5))
    for state in LEAF_STATES:
        ss = sub[sub[col_state] == state]
        ax.scatter(ss[x], ss[y], s=22, alpha=0.65, c=STATE_COLORS[state], label=f'{state} (n={len(ss)})', edgecolors='none')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title + (f' (OM{om_id})' if om_id is not None else ''), fontsize=11, fontweight='bold')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


# Global diagnostic scatters (raw features)
_scatter_by_state(pred_df, 'veg_fraction_hsv', 'yellow_fraction_hsv', 'Green vs Yellow fraction')
_scatter_by_state(pred_df, 'gcc_mean', 'rcc_mean', 'GCC vs RCC')
_scatter_by_state(pred_df, 'hsv_v_mean', 'hsv_s_mean', 'HSV Value vs Saturation')
_scatter_by_state(pred_df, 'gray_entropy', 'gray_std_smooth', 'Entropy vs Gray std (texture)')

In [ ]:
def show_tree_chips(chain_id: int, df: pd.DataFrame, manifest_df: pd.DataFrame, use_smooth: bool = True, max_cols: int = 7):
    """Grid of OM chips for one tree with predicted label overlays."""
    col_state = 'leaf_state_smooth' if use_smooth else 'leaf_state_raw'
    g = df[df['chain_id'] == int(chain_id)].sort_values('om_id')
    if g.empty:
        print('No rows found for chain_id:', chain_id)
        return
    # Resolve chip paths from manifest
    m = manifest_df[(manifest_df['chain_id'] == int(chain_id)) & (manifest_df['status'] == 'ok')].copy()
    m = m.sort_values('om_id')
    if m.empty:
        print('No crop images in manifest for chain_id:', chain_id)
        return
    om_list = m['om_id'].astype(int).tolist()
    n = len(om_list)
    n_cols = min(max_cols, n)
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 2.9 * n_rows), squeeze=False)
    axes = axes.ravel()
    for i, (om_id, crop_path) in enumerate(zip(om_list, m['crop_path'].tolist())):
        ax = axes[i]
        img = read_rgb_u8(crop_path)
        if img is not None:
            ax.imshow(img)
        ax.axis('off')
        row = g[g['om_id'] == int(om_id)].iloc[0] if (g['om_id'] == int(om_id)).any() else None
        if row is not None:
            st = str(row[col_state])
            conf = float(row['confidence_smooth'] if use_smooth else row['confidence_raw'])
            ax.set_title(f'OM{om_id}\n{st}\nconf={conf:.2f}', fontsize=8, color=STATE_COLORS.get(st, 'white'))
        else:
            ax.set_title(f'OM{om_id}', fontsize=8)
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    fig.suptitle(f'Tree(chain_id)={chain_id} — chip sequence', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def sample_trees_for_audit(df: pd.DataFrame, n_each: int = 6, use_smooth: bool = True, seed: int = 7) -> List[int]:
    rng = np.random.default_rng(seed)
    col_state = 'leaf_state_smooth' if use_smooth else 'leaf_state_raw'
    tree_ids = sorted(df['chain_id'].unique().tolist())
    picks: List[int] = []
    for state in LEAF_STATES:
        # pick trees that contain this state at least once
        cand = df[df[col_state] == state]['chain_id'].unique().tolist()
        if not cand:
            continue
        cand = sorted(cand)
        choose = rng.choice(cand, size=min(n_each, len(cand)), replace=False).tolist()
        picks.extend([int(x) for x in choose])
    return sorted(set(picks))


audit_ids = sample_trees_for_audit(pred_df, n_each=4, use_smooth=True, seed=3)
print('Audit tree_ids:', audit_ids)
for tid in audit_ids[:8]:
    show_tree_chips(tid, pred_df, manifest_tree, use_smooth=True, max_cols=7)

In [ ]:
# Borderline cases: lowest-confidence smoothed labels (good for tuning)
low_conf = (
    pred_df.sort_values('confidence_smooth', ascending=True)
    .drop_duplicates(['chain_id'])
    .head(12)
    .copy()
 )
print('Lowest-confidence trees (by best-path state prob):')
display(low_conf[['chain_id','confidence_smooth']])

for tid in low_conf['chain_id'].tolist()[:6]:
    show_tree_chips(int(tid), pred_df, manifest_tree, use_smooth=True, max_cols=7)